# Data Synthesis notebook

In [8]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

import pickle
from logging import INFO, ERROR
from typing import Any

from hydra import initialize, compose
from omegaconf import OmegaConf

from midst_toolkit.common.config import ClavaDDPMMatchingConfig, ClavaDDPMSamplingConfig, GeneralConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER, log
from midst_toolkit.models.clavaddpm.data_loaders import load_tables
from midst_toolkit.models.clavaddpm.enumerations import Relation
from midst_toolkit.models.clavaddpm.synthesizer import clava_synthesizing


## Step 1: Load the config

In [9]:
# Context manager ensures global state is cleaned up after initialization
with initialize(version_base=None, config_path="."):
    # Load config.yaml and pass optional command-line style overrides
    cfg = compose(config_name="config")

# View the configuration as a standard YAML string
print(OmegaConf.to_yaml(cfg))

diffusion_config:
  d_layers:
  - 512
  - 512
  dropout: 0.0
  num_timesteps: 2
  model_type: mlp
  iterations: 2
  batch_size: 4
  lr: 0.0006
  gaussian_loss_type: mse
  weight_decay: 1.0e-05
  scheduler: cosine
  data_split_ratios:
  - 0.99
  - 0.005
  - 0.005
sampling_config:
  batch_size: 1
  classifier_scale: 1.0
sample_scale: 0.2
matching_config:
  num_matching_clusters: 1
  matching_batch_size: 1000
  unique_matching: true
  no_matching: false



## Step 2: load the trained model

In [10]:
ROOT = Path.cwd()
REFERENEC_ROOT = ROOT / "implementations" / "tabular_data" / "single_table" 
# Set data and output directories
base_data_dir = REFERENEC_ROOT / "data"
base_output_dir = REFERENEC_ROOT / "results"

Load the data and `dataset_meta.json` that should be provided in `base_data_dire` directory. Since, this is a single-table synthesis example, we expect that `relation_order` in `dataset_meta.json` include only one table.

In [11]:
log(INFO, f"Checking for a pre-trained model in {base_output_dir}...")
# Load single table data and relation order
tables, relation_order, _ = load_tables(base_data_dir)

assert len(relation_order) == 1 and relation_order[0][0] is None, (
    "Relation order is not configured for single-table. "
    "For multi-table synthesizing, please use the multi-table example. "
    f"Relation order: {relation_order}"
)

INFO :      Checking for a pre-trained model in /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/single_table/results...
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (16000, 8)
INFO :      Total dataframe shape: (16000, 8)
INFO :      Numerical data shape: (16000, 4)
INFO :      Categorical data shape: (16000, 4)


The following code loads the trained model from `base_output_dir` that is saved in the previous step. This code works for multi-table and single-table, and loads the models based on the provided table relation order in `dataset_meta.json`.  

In the single-table example, relation order consists of only one table, with no parent (`null`).

In [12]:
model_file_paths: dict[Relation, dict[str, Any]] = {}
for relation in relation_order:
    model_file_path = Path(base_output_dir) / "models" / f"{relation[0]}_{relation[1]}_ckpt.pkl"
    model_file_paths[relation] = {
        "file_path": model_file_path,
        "exists": model_file_path.exists(),
    }

if all(result["exists"] for result in model_file_paths.values()):
    log(INFO, f"Found previous results in {base_output_dir}.")
    log(INFO, "Loading models...")
    models = {}
    for relation in relation_order:
        with open(model_file_paths[relation]["file_path"], "rb") as f:
            models[relation] = pickle.load(f)
    log(INFO, "All models loaded successfully.")
else:
    log(INFO, "Not all previous results found. Training a new model from scratch.")
    log(INFO, f"Summary of results: {model_file_paths}")
    log(ERROR, "Not all models found. Run the training script first.")



INFO :      Found previous results in /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/single_table/results.
INFO :      Loading models...
INFO :      All models loaded successfully.


## Synthesize data

In [13]:
log(INFO, f"Synthesizing data...")

clava_synthesizing(
    tables,
    relation_order,
    Path(base_output_dir),
    models,
    GeneralConfig(data_dir=base_data_dir, test_data_dir=base_data_dir, exp_name="single_table_synthesizing", workspace_dir=base_output_dir, sample_prefix=""),
    ClavaDDPMSamplingConfig(**cfg.sampling_config),
    ClavaDDPMMatchingConfig(**cfg.matching_config),
    sample_scale=cfg.sample_scale
)

log(INFO, "Data synthesized successfully.")

INFO :      Synthesizing data...
INFO :      Generating None -> trans
INFO :      Sample size: 3200
INFO :      Data synthesized successfully.
